# 5.7 — Official FastTD3 on MuscleMimic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

One cell: install MyoSuite, then the FastTD3 trainer from `training/fasttd3/` ([younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3)), then smoke-train `myoMimicFullbody-v0`.

- **CPU** (`SMOKE = True`): Gymnasium backend. No Hugging Face token, no Warp.
- **GPU** (`SMOKE = False`): mjlab / Warp twin. Runtime → T4 first, then re-run.

Keep `SMOKE = True` until the cell prints `ok`.

In [ ]:
from __future__ import annotations

SMOKE = True  # CPU plumbing. False = mjlab train (needs CUDA).
ENV_ID = "myoMimicFullbody-v0"
TRAIN_STEPS = 50_000
N_ENVS = 1024

import os
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/MyoHub/myosuite.git"
GIT_REF = "tutorial/fasttd3-directional-scratch"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )
    dest = Path("/content/myosuite")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(dest), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    else:
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{dest}[musclemimic]"]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(dest / "training" / "fasttd3")]
    )
    repo = dest
else:
    here = Path.cwd().resolve()
    repo = next((d for d in (here, *here.parents) if (d / "myosuite" / "__init__.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Run this notebook from a myosuite clone, or open it in Colab.")
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    pkg = repo / "training" / "fasttd3"
    if str(pkg) not in sys.path:
        sys.path.insert(0, str(pkg))

import torch

out = repo / "runs" / "fasttd3"
cmd = [sys.executable, "-m", "myosuite_fasttd3", ENV_ID, "--out", str(out)]
if SMOKE:
    cmd += ["--smoke", "--backend", "cpu", "--device", "cpu"]
else:
    if not torch.cuda.is_available():
        raise SystemExit("No CUDA. Runtime → T4/A100, or keep SMOKE=True.")
    if IN_COLAB:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "mjlab>=1.4.0",
                "mujoco-warp>=3.7.0",
                "warp-lang>=1.12.0",
            ]
        )
    cmd += ["--backend", "mjlab", "--num-envs", str(N_ENVS), "--total-timesteps", str(TRAIN_STEPS)]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd, cwd=str(repo))
print("ok")
